# Pipeline Multilabel Classification dengan Seleksi Fitur Statistik dan Metaheuristik

Notebook ini disusun agar dapat dijalankan dari atas ke bawah tanpa error urutan variabel. Alur utama:

1. Dataset `cleaned_df.csv` dibaca dengan aman.
2. Baris kosong dihapus lebih awal, sehingga data valid tidak memakai baris kosong berlebih.
3. Label multilabel dibersihkan menjadi format biner 0/1.
4. Preprocessing teks dibuat sebelum train-test split.
5. `X_train_text`, `X_test_text`, `Y_train`, dan `Y_test` dibuat sebelum embedding.
6. `BertEmbedder` dibuat sebelum `X_train_emb`.
7. `mlsmote()` dibuat sebelum `X_train_bal`.
8. Seleksi fitur statistik dijalankan setelah MLSMOTE: Fisher Score, Variance Threshold, Mean Absolute Difference, Dispersion Ratio, ANOVA, Pearson's Correlation, Mutual Information, dan ReliefF.
9. Metode statistik terbaik diproses lagi menggunakan metaheuristik: PSO, GWO, GA, dan SA.
10. Random Forest multilabel digunakan untuk evaluasi setiap metode.
11. Hasil ditampilkan dalam tabel berisi Precision, Recall, F1 score, dan Accuracy.

Catatan: Precision, Recall, dan F1 score memakai rata-rata `weighted`. Accuracy memakai subset accuracy multilabel.


In [ ]:
# ============================================================
# 1. Instalasi library
# ============================================================
%pip install -q gdown transformers Sastrawi tqdm scikit-learn matplotlib joblib


In [ ]:
# ============================================================
# 2. Import library dan konfigurasi utama
# ============================================================
import os
import re
import random
import hashlib
import warnings
from pathlib import Path

import gdown
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split, KFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import VarianceThreshold, f_classif, mutual_info_classif
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import accuracy_score, hamming_loss, precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

# Nama file
DATA_PATH = Path("cleaned_df.csv")
ACRONYM_PATH = Path("acronym.txt")
KBBA_PATH = Path("kbba.txt")
STOPWORDS_PATH = Path("stopwordsID.txt")

# Kolom dataset
TEXT_COL = "saran"
CLEAN_TEXT_COL = "text_clean"
LABELS = ["SU", "DPAL", "DSTI", "DA", "DPMBKA", "DK", "Perpustakaan"]

# Konfigurasi embedding
MODEL_NAME = "indobenchmark/indobert-base-p1"
MAX_LEN = 128
BATCH_SIZE = 16
TEST_SIZE = 0.20
CACHE_DIR = Path("cache_embedding")
CACHE_DIR.mkdir(exist_ok=True)

# Konfigurasi seleksi fitur statistik
FEATURE_TOP_K = 200          # jumlah fitur teratas untuk metode berbasis skor
THETA_VT = 0.0               # threshold Variance Threshold
RELIEF_SAMPLE_SIZE = 400     # batasi sampel ReliefF agar proses ringan

# Konfigurasi Random Forest evaluasi
RF_N_ESTIMATORS = 120
RF_MAX_DEPTH = 12

# Konfigurasi fitness dan metaheuristik
CV_SPLITS = 3
META_POP_SIZE = 10
META_MAX_ITER = 10
META_PATIENCE = 4
ALPHA = 1.0       # bobot weighted F1
BETA = 0.5        # penalti hamming loss
GAMMA = 0.5       # bobot recall label minoritas

print("Konfigurasi selesai.")


In [ ]:
# ============================================================
# 3. Download dataset dan file pendukung jika belum tersedia
# ============================================================
FILES = {
    "acronym.txt": "1i-_dpAtulIM2gdfk6mXbfi_tfoqWz036",
    "cleaned_df.csv": "1Dtp7emDjco9BeKbZPS6P5ImQXVmNPGyM",
    "kbba.txt": "1OeRk-QnxogCvTGZiYN2hHYqqMpUuFYhk",
    "stopwordsID.txt": "1_8NjGVBjnx_7OrjvjgFAM2wvnJSNK6qf",
}


def download_if_missing(filename, file_id):
    path = Path(filename)
    if path.exists() and path.stat().st_size > 0:
        print(f"File sudah ada: {filename}")
        return path

    print(f"Mengunduh {filename} ...")
    url = f"https://drive.google.com/uc?id={file_id}"
    gdown.download(url, str(path), quiet=False)

    if not path.exists() or path.stat().st_size == 0:
        raise FileNotFoundError(
            f"{filename} belum tersedia. Upload file ini ke folder kerja atau cek akses Google Drive."
        )
    return path


for filename, file_id in FILES.items():
    download_if_missing(filename, file_id)

print("Semua file utama sudah tersedia.")


In [ ]:
# ============================================================
# 4. Fungsi preprocessing teks
# ============================================================
try:
    from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
    stemmer = StemmerFactory().create_stemmer()
except Exception:
    stemmer = None


def load_dictionary(path, separator, skip_first=False):
    data = {}
    path = Path(path)
    if not path.exists():
        return data

    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()

    if skip_first and len(lines) > 0:
        lines = lines[1:]

    for line in lines:
        line = line.strip()
        if not line or separator not in line:
            continue
        key, value = line.split(separator, 1)
        key = key.strip().lower()
        value = value.strip().lower()
        if key and value:
            data[key] = value
    return data


def load_stopwords(path):
    path = Path(path)
    if not path.exists():
        return {
            "yang", "dan", "di", "ke", "dari", "ini", "itu", "untuk",
            "dengan", "pada", "adalah", "karena", "atau", "sebagai",
            "dalam", "saya", "kami", "mereka", "ada", "agar"
        }

    words = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            word = line.strip().lower()
            if word:
                words.append(word)

    stop_words = set(words)
    # Kata ini penting untuk konteks keluhan, jadi tidak dihapus.
    stop_words = stop_words - {"kurang", "banyak", "tidak", "belum", "jangan"}
    return stop_words


ACRONYM = load_dictionary(ACRONYM_PATH, "=")
SLANG = load_dictionary(KBBA_PATH, "\t")
STOP_WORDS = load_stopwords(STOPWORDS_PATH)


def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"@\w+|#\w+", " ", text)
    text = re.sub(r"[^a-zA-ZÀ-ÿ\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = text.split()
    tokens = [ACRONYM.get(token, token) for token in tokens]
    tokens = [SLANG.get(token, token) for token in tokens]
    tokens = [token for token in tokens if token not in STOP_WORDS]

    if stemmer is not None:
        tokens = [stemmer.stem(token) for token in tokens]

    return " ".join(tokens).strip()


print("Jumlah acronym:", len(ACRONYM))
print("Jumlah slang   :", len(SLANG))
print("Jumlah stopword:", len(STOP_WORDS))


In [ ]:
# ============================================================
# 5. Load dataset, hapus baris kosong, dan validasi label
# ============================================================
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset tidak ditemukan: {DATA_PATH}")

df = pd.read_csv(DATA_PATH, low_memory=False)
df.columns = df.columns.str.strip()

required_cols = [TEXT_COL] + LABELS
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"Kolom berikut tidak ditemukan dalam dataset: {missing_cols}")

print("Jumlah baris awal:", len(df))

# Hapus baris tanpa teks. Ini mencegah baris kosong ikut masuk ke embedding.
df = df.dropna(subset=[TEXT_COL]).copy()
df[TEXT_COL] = df[TEXT_COL].astype(str).str.strip()
df = df[df[TEXT_COL].ne("")].copy()

# Bersihkan label agar selalu numerik biner 0/1.
for label in LABELS:
    df[label] = pd.to_numeric(df[label], errors="coerce")

df[LABELS] = df[LABELS].replace([np.inf, -np.inf], np.nan).fillna(0)
df[LABELS] = df[LABELS].clip(0, 1).astype(int)

# Preprocessing teks dilakukan sebelum split agar kolom yang dipakai konsisten.
df[CLEAN_TEXT_COL] = df[TEXT_COL].apply(clean_text)
df = df[df[CLEAN_TEXT_COL].str.strip().ne("")].copy()
df = df.reset_index(drop=True)

print("Jumlah baris valid setelah pembersihan:", len(df))
print("Shape dataset:", df.shape)
print("\nDistribusi label:")
print(df[LABELS].sum().sort_values(ascending=False))

display(df[[TEXT_COL, CLEAN_TEXT_COL] + LABELS].head())


In [ ]:
# ============================================================
# 6. Visualisasi distribusi label
# ============================================================
label_counts = df[LABELS].sum().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
bars = plt.bar(label_counts.index, label_counts.values)
plt.title("Distribusi Label pada Dataset Multilabel")
plt.xlabel("Label")
plt.ylabel("Jumlah Data")
plt.xticks(rotation=45)

for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        int(height),
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 7. Train-test split yang aman untuk multilabel
# ============================================================
def safe_multilabel_split(X, Y, test_size=0.2, random_state=42, max_trials=100):
    # Mencari split yang tetap memiliki minimal satu data positif untuk setiap label pada train dan test.
    for seed in range(random_state, random_state + max_trials):
        X_train, X_test, Y_train, Y_test = train_test_split(
            X,
            Y,
            test_size=test_size,
            random_state=seed,
            shuffle=True
        )
        if (Y_train.sum(axis=0) > 0).all() and (Y_test.sum(axis=0) > 0).all():
            print(f"Split aman ditemukan dengan random_state={seed}")
            return X_train, X_test, Y_train, Y_test, seed

    print("Peringatan: split aman tidak ditemukan. Menggunakan random_state awal.")
    X_train, X_test, Y_train, Y_test = train_test_split(
        X,
        Y,
        test_size=test_size,
        random_state=random_state,
        shuffle=True
    )
    return X_train, X_test, Y_train, Y_test, random_state


X_text = df[CLEAN_TEXT_COL].astype(str).values
Y = df[LABELS].astype(int).values

X_train_text, X_test_text, Y_train, Y_test, SPLIT_RANDOM_STATE = safe_multilabel_split(
    X_text,
    Y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

print("Jumlah data train:", len(X_train_text))
print("Jumlah data test :", len(X_test_text))
print("Shape Y_train    :", Y_train.shape)
print("Shape Y_test     :", Y_test.shape)

print("\nDistribusi label train:")
print(pd.Series(Y_train.sum(axis=0), index=LABELS).sort_values(ascending=False))
print("\nDistribusi label test:")
print(pd.Series(Y_test.sum(axis=0), index=LABELS).sort_values(ascending=False))


In [ ]:
# ============================================================
# 8. Ekstraksi embedding: IndoBERT dengan fallback TF-IDF
# ============================================================
try:
    import torch
    from transformers import AutoTokenizer, AutoModel
except Exception as e:
    torch = None
    AutoTokenizer = None
    AutoModel = None
    print("Torch/Transformers belum dapat digunakan:", repr(e))


class BertEmbedder:
    def __init__(self, model_name, max_len=128, device=None):
        if torch is None or AutoTokenizer is None or AutoModel is None:
            raise ImportError("Torch atau Transformers tidak tersedia.")

        self.model_name = model_name
        self.max_len = max_len
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")

        print("Device yang digunakan:", self.device)
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.model.to(self.device)
        self.model.eval()

    def encode(self, texts, batch_size=16):
        texts = [str(text) for text in texts]
        embeddings = []

        for i in tqdm(range(0, len(texts), batch_size), desc="Extracting BERT embeddings"):
            batch_texts = texts[i:i + batch_size]
            encoded = self.tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=self.max_len,
                return_tensors="pt"
            )
            encoded = {key: value.to(self.device) for key, value in encoded.items()}

            with torch.no_grad():
                outputs = self.model(**encoded)
                cls_embedding = outputs.last_hidden_state[:, 0, :]

            embeddings.append(cls_embedding.cpu().numpy())

        return np.vstack(embeddings).astype(np.float32)


def make_cache_name(prefix, texts, model_name):
    sample = " ".join([str(x) for x in texts[:30]])
    raw_key = f"{prefix}_{len(texts)}_{model_name}_{SPLIT_RANDOM_STATE}_{sample}"
    digest = hashlib.md5(raw_key.encode("utf-8")).hexdigest()[:10]
    safe_model = re.sub(r"[^a-zA-Z0-9]+", "_", model_name).strip("_")
    return f"{prefix}_{safe_model}_{len(texts)}_{digest}"


def encode_with_resume(texts, embedder, name, batch_size=16, chunk_size=500, cache_dir=CACHE_DIR):
    texts = [str(text) for text in texts]
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(exist_ok=True)

    final_path = cache_dir / f"{name}_final.npy"
    if final_path.exists():
        embeddings = np.load(final_path)
        if embeddings.shape[0] == len(texts):
            print(f"Loading final embedding: {final_path}")
            return embeddings
        print("Cache final tidak cocok dengan jumlah data saat ini. Embedding dibuat ulang.")
        final_path.unlink()

    total_chunks = (len(texts) + chunk_size - 1) // chunk_size
    chunk_paths = []

    for i in tqdm(range(total_chunks), desc=f"Encoding {name}"):
        start = i * chunk_size
        end = min((i + 1) * chunk_size, len(texts))
        chunk_path = cache_dir / f"{name}_chunk_{i}.npy"
        temp_path = chunk_path.with_suffix(".tmp.npy")
        chunk_paths.append(chunk_path)

        if chunk_path.exists():
            chunk = np.load(chunk_path)
            if chunk.shape[0] == (end - start):
                continue
            print(f"Chunk tidak cocok dan dibuat ulang: {chunk_path}")
            chunk_path.unlink()

        print(f"Processing {name} chunk {i + 1}/{total_chunks} | data {start}:{end}")
        emb_chunk = embedder.encode(texts[start:end], batch_size=batch_size)
        np.save(temp_path, emb_chunk)
        os.replace(temp_path, chunk_path)

    embeddings = [np.load(path) for path in chunk_paths]
    embeddings = np.vstack(embeddings).astype(np.float32)
    np.save(final_path, embeddings)
    print(f"Final embedding saved: {final_path}")
    return embeddings


def build_embeddings(X_train_text, X_test_text):
    # Memakai IndoBERT jika tersedia. Jika gagal, memakai TF-IDF agar pipeline tetap berjalan.
    tfidf_vectorizer = None

    try:
        embedder = BertEmbedder(model_name=MODEL_NAME, max_len=MAX_LEN)
        train_cache_name = make_cache_name("train", X_train_text, MODEL_NAME)
        test_cache_name = make_cache_name("test", X_test_text, MODEL_NAME)

        X_train_emb = encode_with_resume(
            X_train_text,
            embedder,
            train_cache_name,
            batch_size=BATCH_SIZE,
            chunk_size=500
        )
        X_test_emb = encode_with_resume(
            X_test_text,
            embedder,
            test_cache_name,
            batch_size=BATCH_SIZE,
            chunk_size=500
        )
        embedding_backend = "IndoBERT_CLS"

    except Exception as e:
        print("IndoBERT tidak dapat digunakan. Pipeline dilanjutkan dengan TF-IDF.")
        print("Penyebab:", repr(e))
        tfidf_vectorizer = TfidfVectorizer(max_features=768, ngram_range=(1, 2), min_df=1)
        X_train_emb = tfidf_vectorizer.fit_transform(X_train_text).toarray().astype(np.float32)
        X_test_emb = tfidf_vectorizer.transform(X_test_text).toarray().astype(np.float32)
        embedding_backend = "TFIDF_fallback"

    return X_train_emb, X_test_emb, embedding_backend, tfidf_vectorizer


X_train_emb, X_test_emb, EMBEDDING_BACKEND, TFIDF_VECTORIZER = build_embeddings(
    list(X_train_text),
    list(X_test_text)
)

print("Embedding backend:", EMBEDDING_BACKEND)
print("Shape X_train_emb:", X_train_emb.shape)
print("Shape X_test_emb :", X_test_emb.shape)


In [ ]:
# ============================================================
# 9. MLSMOTE hanya pada data training
# ============================================================
def mlsmote(X, Y, k_neighbors=5, random_state=42):
    # Implementasi sederhana MLSMOTE untuk data multilabel.
    rng = np.random.default_rng(random_state)

    X = np.asarray(X, dtype=np.float32)
    Y = np.asarray(Y).astype(int)

    label_counts = Y.sum(axis=0)
    max_count = int(label_counts.max())

    synthetic_X = []
    synthetic_Y = []

    for label_idx in range(Y.shape[1]):
        current_count = int(label_counts[label_idx])

        if current_count == 0 or current_count >= max_count:
            continue

        minority_indices = np.where(Y[:, label_idx] == 1)[0]
        if len(minority_indices) < 2:
            continue

        X_min = X[minority_indices]
        Y_min = Y[minority_indices]

        n_neighbors = min(k_neighbors + 1, len(minority_indices))
        nn = NearestNeighbors(n_neighbors=n_neighbors)
        nn.fit(X_min)
        _, neighbor_indices = nn.kneighbors(X_min)

        n_to_generate = max_count - current_count

        for _ in range(n_to_generate):
            seed_pos = rng.integers(0, len(minority_indices))
            available_neighbors = neighbor_indices[seed_pos][1:]
            if len(available_neighbors) == 0:
                continue

            neighbor_pos = rng.choice(available_neighbors)
            x_seed = X_min[seed_pos]
            x_neighbor = X_min[neighbor_pos]

            lam = rng.random()
            x_new = x_seed + lam * (x_neighbor - x_seed)

            related_positions = np.append(available_neighbors, seed_pos)
            y_votes = Y_min[related_positions].mean(axis=0)
            y_new = (y_votes >= 0.5).astype(int)
            y_new[label_idx] = 1

            synthetic_X.append(x_new)
            synthetic_Y.append(y_new)

    if len(synthetic_X) == 0:
        return X, Y

    X_new = np.vstack([X, np.asarray(synthetic_X, dtype=np.float32)])
    Y_new = np.vstack([Y, np.asarray(synthetic_Y, dtype=int)])
    return X_new, Y_new


X_train_bal, Y_train_bal = mlsmote(
    X_train_emb,
    Y_train,
    k_neighbors=5,
    random_state=RANDOM_STATE
)

print("Before MLSMOTE:", X_train_emb.shape, Y_train.shape)
print("After MLSMOTE :", X_train_bal.shape, Y_train_bal.shape)
print("\nDistribusi label setelah MLSMOTE:")
print(pd.Series(Y_train_bal.sum(axis=0), index=LABELS).sort_values(ascending=False))


## Tahap 10

Mulai dari cell ini, seluruh seleksi fitur dijalankan setelah MLSMOTE. Artinya, data training yang digunakan adalah `X_train_bal` dan `Y_train_bal`, sedangkan data testing tetap memakai `X_test_emb` dan `Y_test` agar evaluasi tetap objektif.


In [ ]:
# ============================================================
# 10. Fungsi evaluasi Random Forest multilabel
# ============================================================
def compute_main_metrics(Y_true, Y_pred):
    return {
        "Precision": precision_score(Y_true, Y_pred, average="weighted", zero_division=0),
        "Recall": recall_score(Y_true, Y_pred, average="weighted", zero_division=0),
        "F1 score": f1_score(Y_true, Y_pred, average="weighted", zero_division=0),
        "Accuracy": accuracy_score(Y_true, Y_pred),
        "Hamming Loss": hamming_loss(Y_true, Y_pred)
    }


def train_evaluate_rf(X_train, Y_train, X_test, Y_test, random_state=42):
    base_rf = RandomForestClassifier(
        n_estimators=RF_N_ESTIMATORS,
        max_depth=RF_MAX_DEPTH,
        class_weight="balanced_subsample",
        random_state=random_state,
        n_jobs=-1
    )
    clf = MultiOutputClassifier(base_rf)
    clf.fit(X_train, Y_train)
    Y_pred = clf.predict(X_test)
    metrics = compute_main_metrics(Y_test, Y_pred)
    return metrics, clf, Y_pred


def format_results_table(rows):
    df_result = pd.DataFrame(rows)
    metric_cols = ["Precision", "Recall", "F1 score", "Accuracy"]
    for col in metric_cols:
        df_result[col] = df_result[col].astype(float).round(4)
    if "Hamming Loss" in df_result.columns:
        df_result["Hamming Loss"] = df_result["Hamming Loss"].astype(float).round(4)
    return df_result


print("Fungsi evaluasi siap digunakan.")


In [ ]:
# ============================================================
# 11. Fungsi seleksi fitur statistik untuk multilabel
# ============================================================
def sanitize_scores(scores):
    scores = np.asarray(scores, dtype=np.float64)
    scores = np.nan_to_num(scores, nan=0.0, posinf=0.0, neginf=0.0)
    return scores


def select_top_k(scores, top_k):
    scores = sanitize_scores(scores)
    n_features = len(scores)
    k = min(top_k, n_features)
    if k <= 0:
        raise ValueError("top_k harus lebih dari 0.")
    selected_idx = np.argsort(scores)[::-1][:k]
    selected_idx = np.sort(selected_idx)
    mask = np.zeros(n_features, dtype=bool)
    mask[selected_idx] = True
    return mask, selected_idx, scores


def fisher_score_multilabel(X, Y, eps=1e-12):
    X = np.asarray(X, dtype=np.float64)
    Y = np.asarray(Y).astype(int)
    scores = []

    for label_idx in range(Y.shape[1]):
        y = Y[:, label_idx]
        if len(np.unique(y)) < 2:
            continue

        X_pos = X[y == 1]
        X_neg = X[y == 0]
        mu_all = X.mean(axis=0)
        mu_pos = X_pos.mean(axis=0)
        mu_neg = X_neg.mean(axis=0)
        var_pos = X_pos.var(axis=0)
        var_neg = X_neg.var(axis=0)

        numerator = len(X_pos) * (mu_pos - mu_all) ** 2 + len(X_neg) * (mu_neg - mu_all) ** 2
        denominator = len(X_pos) * var_pos + len(X_neg) * var_neg + eps
        scores.append(numerator / denominator)

    if len(scores) == 0:
        return np.zeros(X.shape[1])
    return sanitize_scores(np.mean(scores, axis=0))


def mean_absolute_difference_multilabel(X, Y):
    X = np.asarray(X, dtype=np.float64)
    Y = np.asarray(Y).astype(int)
    scores = []

    for label_idx in range(Y.shape[1]):
        y = Y[:, label_idx]
        if len(np.unique(y)) < 2:
            continue
        mean_pos = X[y == 1].mean(axis=0)
        mean_neg = X[y == 0].mean(axis=0)
        scores.append(np.abs(mean_pos - mean_neg))

    if len(scores) == 0:
        return np.zeros(X.shape[1])
    return sanitize_scores(np.mean(scores, axis=0))


def dispersion_ratio_multilabel(X, Y, eps=1e-12):
    X = np.asarray(X, dtype=np.float64)
    Y = np.asarray(Y).astype(int)
    scores = []

    for label_idx in range(Y.shape[1]):
        y = Y[:, label_idx]
        if len(np.unique(y)) < 2:
            continue

        X_pos = X[y == 1]
        X_neg = X[y == 0]
        mean_pos = X_pos.mean(axis=0)
        mean_neg = X_neg.mean(axis=0)
        std_pos = X_pos.std(axis=0)
        std_neg = X_neg.std(axis=0)

        cv_pos = std_pos / (np.abs(mean_pos) + eps)
        cv_neg = std_neg / (np.abs(mean_neg) + eps)
        ratio = np.maximum(cv_pos, cv_neg) / (np.minimum(cv_pos, cv_neg) + eps)
        mean_gap = np.abs(mean_pos - mean_neg)
        scores.append(ratio * mean_gap)

    if len(scores) == 0:
        return np.zeros(X.shape[1])
    return sanitize_scores(np.mean(scores, axis=0))


def anova_score_multilabel(X, Y):
    X = np.asarray(X, dtype=np.float64)
    Y = np.asarray(Y).astype(int)
    scores = []

    for label_idx in range(Y.shape[1]):
        y = Y[:, label_idx]
        if len(np.unique(y)) < 2:
            continue
        f_values, _ = f_classif(X, y)
        scores.append(sanitize_scores(f_values))

    if len(scores) == 0:
        return np.zeros(X.shape[1])
    return sanitize_scores(np.mean(scores, axis=0))


def pearson_score_multilabel(X, Y, eps=1e-12):
    X = np.asarray(X, dtype=np.float64)
    Y = np.asarray(Y, dtype=np.float64)

    X_centered = X - X.mean(axis=0)
    X_std = X_centered.std(axis=0) + eps

    scores = []
    for label_idx in range(Y.shape[1]):
        y = Y[:, label_idx]
        if len(np.unique(y)) < 2:
            continue
        y_centered = y - y.mean()
        y_std = y_centered.std() + eps
        corr = np.abs((X_centered.T @ y_centered) / (len(y) * X_std * y_std))
        scores.append(corr)

    if len(scores) == 0:
        return np.zeros(X.shape[1])
    return sanitize_scores(np.mean(scores, axis=0))


def mutual_information_multilabel(X, Y, random_state=42):
    X = np.asarray(X, dtype=np.float64)
    Y = np.asarray(Y).astype(int)
    scores = []

    for label_idx in range(Y.shape[1]):
        y = Y[:, label_idx]
        if len(np.unique(y)) < 2:
            continue
        mi = mutual_info_classif(X, y, discrete_features=False, random_state=random_state)
        scores.append(sanitize_scores(mi))

    if len(scores) == 0:
        return np.zeros(X.shape[1])
    return sanitize_scores(np.mean(scores, axis=0))


def relieff_score_multilabel(X, Y, sample_size=400, random_state=42, eps=1e-12):
    # ReliefF sederhana untuk multilabel dengan rata-rata skor per label.
    # Skor naik jika sampel berbeda label memiliki jarak fitur yang lebih besar daripada sampel satu label.
    # Sampel dibatasi agar proses tetap ringan pada data embedding.
    rng = np.random.default_rng(random_state)
    X = np.asarray(X, dtype=np.float64)
    Y = np.asarray(Y).astype(int)

    n_samples, n_features = X.shape
    m = min(sample_size, n_samples)
    sample_indices = rng.choice(n_samples, size=m, replace=False)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X_scaled = np.nan_to_num(X_scaled, nan=0.0, posinf=0.0, neginf=0.0)

    weights = np.zeros(n_features, dtype=np.float64)
    valid_label_count = 0

    for label_idx in range(Y.shape[1]):
        y = Y[:, label_idx]
        if len(np.unique(y)) < 2:
            continue

        valid_label_count += 1
        pos_idx = np.where(y == 1)[0]
        neg_idx = np.where(y == 0)[0]

        nn_pos = NearestNeighbors(n_neighbors=min(2, len(pos_idx))).fit(X_scaled[pos_idx])
        nn_neg = NearestNeighbors(n_neighbors=min(2, len(neg_idx))).fit(X_scaled[neg_idx])

        for idx in sample_indices:
            xi = X_scaled[idx:idx + 1]

            if y[idx] == 1:
                same_idx = pos_idx
                diff_idx = neg_idx
                nn_same = nn_pos
                nn_diff = nn_neg
            else:
                same_idx = neg_idx
                diff_idx = pos_idx
                nn_same = nn_neg
                nn_diff = nn_pos

            if len(same_idx) < 2 or len(diff_idx) < 1:
                continue

            same_neighbors = nn_same.kneighbors(xi, return_distance=False)[0]
            same_global = same_idx[same_neighbors]
            same_global = same_global[same_global != idx]
            if len(same_global) == 0:
                continue
            hit_idx = same_global[0]

            miss_local = nn_diff.kneighbors(xi, return_distance=False)[0][0]
            miss_idx = diff_idx[miss_local]

            diff_hit = np.abs(X_scaled[idx] - X_scaled[hit_idx])
            diff_miss = np.abs(X_scaled[idx] - X_scaled[miss_idx])
            weights += (diff_miss - diff_hit)

    if valid_label_count == 0:
        return np.zeros(n_features)

    weights = weights / (m * valid_label_count + eps)
    weights = weights - weights.min()
    return sanitize_scores(weights)


def apply_statistical_selector(method_name, X_train, Y_train, X_test, top_k=200):
    if method_name == "Variance Threshold":
        selector = VarianceThreshold(threshold=THETA_VT)
        X_train_selected = selector.fit_transform(X_train)
        X_test_selected = selector.transform(X_test)
        mask = selector.get_support()
        selected_idx = np.where(mask)[0]
        scores = X_train.var(axis=0)
        if X_train_selected.shape[1] == 0:
            raise ValueError("Tidak ada fitur tersisa setelah Variance Threshold.")
        return X_train_selected, X_test_selected, mask, selected_idx, scores

    score_functions = {
        "Fisher Score": fisher_score_multilabel,
        "Mean Absolute Difference": mean_absolute_difference_multilabel,
        "Dispersion Ratio": dispersion_ratio_multilabel,
        "ANOVA": anova_score_multilabel,
        "Pearson's Correlation": pearson_score_multilabel,
        "Mutual Information": lambda X, Y: mutual_information_multilabel(X, Y, random_state=RANDOM_STATE),
        "ReliefF": lambda X, Y: relieff_score_multilabel(
            X,
            Y,
            sample_size=RELIEF_SAMPLE_SIZE,
            random_state=RANDOM_STATE
        ),
    }

    if method_name not in score_functions:
        raise ValueError(f"Metode tidak dikenal: {method_name}")

    scores = score_functions[method_name](X_train, Y_train)
    mask, selected_idx, scores = select_top_k(scores, top_k=top_k)
    X_train_selected = X_train[:, mask]
    X_test_selected = X_test[:, mask]
    return X_train_selected, X_test_selected, mask, selected_idx, scores


STATISTICAL_METHODS = [
    "Fisher Score",
    "Variance Threshold",
    "Mean Absolute Difference",
    "Dispersion Ratio",
    "ANOVA",
    "Pearson's Correlation",
    "Mutual Information",
    "ReliefF"
]

print("Fungsi seleksi fitur statistik siap digunakan.")


In [ ]:
# ============================================================
# 12. Jalankan semua metode seleksi fitur statistik
# ============================================================
statistical_results = []
statistical_artifacts = {}

for method_name in STATISTICAL_METHODS:
    print("\n" + "=" * 70)
    print(f"Menjalankan seleksi fitur statistik: {method_name}")

    X_train_fs, X_test_fs, mask, selected_idx, scores = apply_statistical_selector(
        method_name,
        X_train_bal,
        Y_train_bal,
        X_test_emb,
        top_k=FEATURE_TOP_K
    )

    metrics, model, y_pred = train_evaluate_rf(
        X_train_fs,
        Y_train_bal,
        X_test_fs,
        Y_test,
        random_state=RANDOM_STATE
    )

    row = {
        "Method": method_name,
        "Selected Features": int(X_train_fs.shape[1]),
        "Precision": metrics["Precision"],
        "Recall": metrics["Recall"],
        "F1 score": metrics["F1 score"],
        "Accuracy": metrics["Accuracy"],
        "Hamming Loss": metrics["Hamming Loss"]
    }
    statistical_results.append(row)

    statistical_artifacts[method_name] = {
        "mask": mask,
        "selected_idx": selected_idx,
        "scores": scores,
        "X_train": X_train_fs,
        "X_test": X_test_fs,
        "model": model,
        "y_pred": y_pred,
        "metrics": metrics
    }

    print(row)

statistical_results_df = format_results_table(statistical_results)
statistical_results_df = statistical_results_df.sort_values(
    by=["F1 score", "Accuracy"],
    ascending=False
).reset_index(drop=True)

print("\nTabel hasil seleksi fitur statistik")
display(statistical_results_df[["Method", "Selected Features", "Precision", "Recall", "F1 score", "Accuracy"]])

BEST_STATISTICAL_METHOD = statistical_results_df.loc[0, "Method"]
print("Metode statistik terbaik berdasarkan F1 score:", BEST_STATISTICAL_METHOD)


In [ ]:
# ============================================================
# 13. Siapkan data dari metode statistik terbaik untuk metaheuristik
# ============================================================
best_stat_artifact = statistical_artifacts[BEST_STATISTICAL_METHOD]
X_train_best_stat = best_stat_artifact["X_train"]
X_test_best_stat = best_stat_artifact["X_test"]
base_selected_idx = best_stat_artifact["selected_idx"]

print("Metode statistik terbaik:", BEST_STATISTICAL_METHOD)
print("Shape X_train_best_stat:", X_train_best_stat.shape)
print("Shape X_test_best_stat :", X_test_best_stat.shape)


In [ ]:
# ============================================================
# 14. Fungsi fitness untuk metaheuristik
# ============================================================
def get_minority_label_indices(Y):
    label_counts = Y.sum(axis=0)
    mean_count = label_counts.mean()
    minority_indices = np.where(label_counts < mean_count)[0]
    if len(minority_indices) == 0:
        minority_indices = np.arange(Y.shape[1])
    return minority_indices


def repair_mask(mask, rng):
    mask = np.asarray(mask).astype(bool)
    if mask.sum() == 0:
        mask[rng.integers(0, len(mask))] = True
    return mask


def sigmoid(x):
    x = np.clip(x, -500, 500)
    return 1 / (1 + np.exp(-x))


def fitness_function(
    feature_mask,
    X,
    Y,
    minority_label_indices,
    alpha=1.0,
    beta=0.5,
    gamma=0.5,
    cv_splits=3,
    random_state=42
):
    feature_mask = np.asarray(feature_mask).astype(bool)
    if feature_mask.sum() == 0:
        return -9999.0

    X_selected = X[:, feature_mask]
    n_splits = min(cv_splits, len(Y))
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    f1_scores = []
    hamming_losses = []
    minority_recalls = []

    for train_idx, val_idx in kf.split(X_selected):
        X_train_fold = X_selected[train_idx]
        X_val_fold = X_selected[val_idx]
        Y_train_fold = Y[train_idx]
        Y_val_fold = Y[val_idx]

        base_rf = RandomForestClassifier(
            n_estimators=60,
            max_depth=10,
            class_weight="balanced_subsample",
            random_state=random_state,
            n_jobs=-1
        )
        clf = MultiOutputClassifier(base_rf)
        clf.fit(X_train_fold, Y_train_fold)
        Y_pred_fold = clf.predict(X_val_fold)

        f1_scores.append(f1_score(Y_val_fold, Y_pred_fold, average="weighted", zero_division=0))
        hamming_losses.append(hamming_loss(Y_val_fold, Y_pred_fold))

        recall_each_label = recall_score(Y_val_fold, Y_pred_fold, average=None, zero_division=0)
        minority_recalls.append(np.mean(recall_each_label[minority_label_indices]))

    fitness = (
        alpha * np.mean(f1_scores)
        - beta * np.mean(hamming_losses)
        + gamma * np.mean(minority_recalls)
    )
    return float(fitness)


print("Fungsi fitness metaheuristik siap digunakan.")


In [ ]:
# ============================================================
# 15. Metaheuristik 1: Particle Swarm Optimization (PSO)
# ============================================================
def pso_feature_selection(
    X,
    Y,
    pop_size=10,
    max_iter=10,
    w=0.729,
    c1=1.496,
    c2=1.496,
    random_state=42,
    patience=4
):
    rng = np.random.default_rng(random_state)
    n_features = X.shape[1]
    minority_label_indices = get_minority_label_indices(Y)

    positions = rng.integers(0, 2, size=(pop_size, n_features)).astype(bool)
    velocities = rng.normal(0, 1, size=(pop_size, n_features))
    positions = np.array([repair_mask(pos, rng) for pos in positions])

    pbest_positions = positions.copy()
    pbest_fitness = np.array([
        fitness_function(pos, X, Y, minority_label_indices, ALPHA, BETA, GAMMA, CV_SPLITS, random_state)
        for pos in pbest_positions
    ])

    best_idx = int(np.argmax(pbest_fitness))
    gbest_position = pbest_positions[best_idx].copy()
    gbest_fitness = float(pbest_fitness[best_idx])
    no_improvement = 0

    for iteration in range(max_iter):
        improved = False
        for j in range(pop_size):
            r1 = rng.random(n_features)
            r2 = rng.random(n_features)
            velocities[j] = (
                w * velocities[j]
                + c1 * r1 * (pbest_positions[j].astype(int) - positions[j].astype(int))
                + c2 * r2 * (gbest_position.astype(int) - positions[j].astype(int))
            )
            probabilities = sigmoid(velocities[j])
            positions[j] = rng.random(n_features) < probabilities
            positions[j] = repair_mask(positions[j], rng)

            current_fitness = fitness_function(
                positions[j], X, Y, minority_label_indices, ALPHA, BETA, GAMMA, CV_SPLITS, random_state
            )
            if current_fitness > pbest_fitness[j]:
                pbest_positions[j] = positions[j].copy()
                pbest_fitness[j] = current_fitness
            if current_fitness > gbest_fitness:
                gbest_position = positions[j].copy()
                gbest_fitness = current_fitness
                improved = True

        no_improvement = 0 if improved else no_improvement + 1
        print(f"PSO iter {iteration + 1:02d} | fitness={gbest_fitness:.4f} | fitur={gbest_position.sum()}")
        if no_improvement >= patience:
            break

    return gbest_position.astype(bool), gbest_fitness


In [ ]:
# ============================================================
# 16. Metaheuristik 2: Grey Wolf Optimizer (GWO)
# ============================================================
def gwo_feature_selection(
    X,
    Y,
    pop_size=10,
    max_iter=10,
    random_state=42,
    patience=4
):
    rng = np.random.default_rng(random_state)
    n_features = X.shape[1]
    minority_label_indices = get_minority_label_indices(Y)

    wolves = rng.integers(0, 2, size=(pop_size, n_features)).astype(bool)
    wolves = np.array([repair_mask(wolf, rng) for wolf in wolves])
    fitness_values = np.array([
        fitness_function(wolf, X, Y, minority_label_indices, ALPHA, BETA, GAMMA, CV_SPLITS, random_state)
        for wolf in wolves
    ])

    best_idx = int(np.argmax(fitness_values))
    best_mask = wolves[best_idx].copy()
    best_fitness = float(fitness_values[best_idx])
    no_improvement = 0

    for iteration in range(max_iter):
        order = np.argsort(fitness_values)[::-1]
        alpha_wolf = wolves[order[0]].astype(float)
        beta_wolf = wolves[order[1 if pop_size > 1 else 0]].astype(float)
        delta_wolf = wolves[order[2 if pop_size > 2 else 0]].astype(float)

        a = 2 - iteration * (2 / max_iter)
        improved = False

        for i in range(pop_size):
            current = wolves[i].astype(float)

            A1 = 2 * a * rng.random(n_features) - a
            C1 = 2 * rng.random(n_features)
            D_alpha = np.abs(C1 * alpha_wolf - current)
            X1 = alpha_wolf - A1 * D_alpha

            A2 = 2 * a * rng.random(n_features) - a
            C2 = 2 * rng.random(n_features)
            D_beta = np.abs(C2 * beta_wolf - current)
            X2 = beta_wolf - A2 * D_beta

            A3 = 2 * a * rng.random(n_features) - a
            C3 = 2 * rng.random(n_features)
            D_delta = np.abs(C3 * delta_wolf - current)
            X3 = delta_wolf - A3 * D_delta

            continuous_position = (X1 + X2 + X3) / 3
            probabilities = sigmoid(continuous_position)
            wolves[i] = rng.random(n_features) < probabilities
            wolves[i] = repair_mask(wolves[i], rng)

            fitness_values[i] = fitness_function(
                wolves[i], X, Y, minority_label_indices, ALPHA, BETA, GAMMA, CV_SPLITS, random_state
            )

            if fitness_values[i] > best_fitness:
                best_mask = wolves[i].copy()
                best_fitness = float(fitness_values[i])
                improved = True

        no_improvement = 0 if improved else no_improvement + 1
        print(f"GWO iter {iteration + 1:02d} | fitness={best_fitness:.4f} | fitur={best_mask.sum()}")
        if no_improvement >= patience:
            break

    return best_mask.astype(bool), best_fitness


In [ ]:
# ============================================================
# 17. Metaheuristik 3: Genetic Algorithm (GA)
# ============================================================
def tournament_selection(population, fitness_values, rng, tournament_size=3):
    candidates = rng.choice(len(population), size=min(tournament_size, len(population)), replace=False)
    best_candidate = candidates[np.argmax(fitness_values[candidates])]
    return population[best_candidate].copy()


def ga_feature_selection(
    X,
    Y,
    pop_size=10,
    max_iter=10,
    crossover_rate=0.8,
    mutation_rate=0.03,
    random_state=42,
    patience=4
):
    rng = np.random.default_rng(random_state)
    n_features = X.shape[1]
    minority_label_indices = get_minority_label_indices(Y)

    population = rng.integers(0, 2, size=(pop_size, n_features)).astype(bool)
    population = np.array([repair_mask(ind, rng) for ind in population])
    fitness_values = np.array([
        fitness_function(ind, X, Y, minority_label_indices, ALPHA, BETA, GAMMA, CV_SPLITS, random_state)
        for ind in population
    ])

    best_idx = int(np.argmax(fitness_values))
    best_mask = population[best_idx].copy()
    best_fitness = float(fitness_values[best_idx])
    no_improvement = 0

    for iteration in range(max_iter):
        new_population = [best_mask.copy()]  # elitism

        while len(new_population) < pop_size:
            parent1 = tournament_selection(population, fitness_values, rng)
            parent2 = tournament_selection(population, fitness_values, rng)

            child1, child2 = parent1.copy(), parent2.copy()
            if rng.random() < crossover_rate:
                crossover_mask = rng.random(n_features) < 0.5
                child1[crossover_mask] = parent2[crossover_mask]
                child2[crossover_mask] = parent1[crossover_mask]

            mutation_mask1 = rng.random(n_features) < mutation_rate
            mutation_mask2 = rng.random(n_features) < mutation_rate
            child1[mutation_mask1] = ~child1[mutation_mask1]
            child2[mutation_mask2] = ~child2[mutation_mask2]

            child1 = repair_mask(child1, rng)
            child2 = repair_mask(child2, rng)
            new_population.extend([child1, child2])

        population = np.array(new_population[:pop_size])
        fitness_values = np.array([
            fitness_function(ind, X, Y, minority_label_indices, ALPHA, BETA, GAMMA, CV_SPLITS, random_state)
            for ind in population
        ])

        current_best_idx = int(np.argmax(fitness_values))
        current_best_fitness = float(fitness_values[current_best_idx])
        improved = current_best_fitness > best_fitness
        if improved:
            best_fitness = current_best_fitness
            best_mask = population[current_best_idx].copy()

        no_improvement = 0 if improved else no_improvement + 1
        print(f"GA iter {iteration + 1:02d} | fitness={best_fitness:.4f} | fitur={best_mask.sum()}")
        if no_improvement >= patience:
            break

    return best_mask.astype(bool), best_fitness


In [ ]:
# ============================================================
# 18. Metaheuristik 4: Simulated Annealing (SA)
# ============================================================
def sa_feature_selection(
    X,
    Y,
    max_iter=10,
    initial_temperature=1.0,
    cooling_rate=0.85,
    mutation_rate=0.05,
    random_state=42,
    patience=4
):
    rng = np.random.default_rng(random_state)
    n_features = X.shape[1]
    minority_label_indices = get_minority_label_indices(Y)

    current_mask = rng.integers(0, 2, size=n_features).astype(bool)
    current_mask = repair_mask(current_mask, rng)
    current_fitness = fitness_function(
        current_mask, X, Y, minority_label_indices, ALPHA, BETA, GAMMA, CV_SPLITS, random_state
    )

    best_mask = current_mask.copy()
    best_fitness = float(current_fitness)
    temperature = initial_temperature
    no_improvement = 0

    for iteration in range(max_iter):
        candidate_mask = current_mask.copy()
        flip_mask = rng.random(n_features) < mutation_rate
        if flip_mask.sum() == 0:
            flip_mask[rng.integers(0, n_features)] = True
        candidate_mask[flip_mask] = ~candidate_mask[flip_mask]
        candidate_mask = repair_mask(candidate_mask, rng)

        candidate_fitness = fitness_function(
            candidate_mask, X, Y, minority_label_indices, ALPHA, BETA, GAMMA, CV_SPLITS, random_state
        )

        delta = candidate_fitness - current_fitness
        accept = delta > 0 or rng.random() < np.exp(delta / max(temperature, 1e-12))

        if accept:
            current_mask = candidate_mask.copy()
            current_fitness = candidate_fitness

        improved = candidate_fitness > best_fitness
        if improved:
            best_mask = candidate_mask.copy()
            best_fitness = float(candidate_fitness)

        temperature *= cooling_rate
        no_improvement = 0 if improved else no_improvement + 1
        print(f"SA iter {iteration + 1:02d} | fitness={best_fitness:.4f} | fitur={best_mask.sum()}")
        if no_improvement >= patience:
            break

    return best_mask.astype(bool), best_fitness


In [ ]:
# ============================================================
# 19. Jalankan PSO, GWO, GA, dan SA pada metode statistik terbaik
# ============================================================
METAHEURISTIC_METHODS = {
    "PSO": lambda X, Y: pso_feature_selection(
        X, Y,
        pop_size=META_POP_SIZE,
        max_iter=META_MAX_ITER,
        random_state=RANDOM_STATE,
        patience=META_PATIENCE
    ),
    "GWO": lambda X, Y: gwo_feature_selection(
        X, Y,
        pop_size=META_POP_SIZE,
        max_iter=META_MAX_ITER,
        random_state=RANDOM_STATE,
        patience=META_PATIENCE
    ),
    "GA": lambda X, Y: ga_feature_selection(
        X, Y,
        pop_size=META_POP_SIZE,
        max_iter=META_MAX_ITER,
        random_state=RANDOM_STATE,
        patience=META_PATIENCE
    ),
    "SA": lambda X, Y: sa_feature_selection(
        X, Y,
        max_iter=META_MAX_ITER,
        random_state=RANDOM_STATE,
        patience=META_PATIENCE
    ),
}

metaheuristic_results = []
metaheuristic_artifacts = {}

for method_name, optimizer in METAHEURISTIC_METHODS.items():
    print("\n" + "=" * 70)
    print(f"Menjalankan metaheuristik: {method_name}")

    meta_mask, meta_fitness = optimizer(X_train_best_stat, Y_train_bal)
    X_train_meta = X_train_best_stat[:, meta_mask]
    X_test_meta = X_test_best_stat[:, meta_mask]

    metrics, model, y_pred = train_evaluate_rf(
        X_train_meta,
        Y_train_bal,
        X_test_meta,
        Y_test,
        random_state=RANDOM_STATE
    )

    selected_idx_after_stat = np.where(meta_mask)[0]
    selected_idx_original = base_selected_idx[selected_idx_after_stat]

    row = {
        "Base Statistical Method": BEST_STATISTICAL_METHOD,
        "Metaheuristic": method_name,
        "Selected Features": int(X_train_meta.shape[1]),
        "Fitness": meta_fitness,
        "Precision": metrics["Precision"],
        "Recall": metrics["Recall"],
        "F1 score": metrics["F1 score"],
        "Accuracy": metrics["Accuracy"],
        "Hamming Loss": metrics["Hamming Loss"]
    }
    metaheuristic_results.append(row)

    metaheuristic_artifacts[method_name] = {
        "mask_after_stat": meta_mask,
        "selected_idx_after_stat": selected_idx_after_stat,
        "selected_idx_original": selected_idx_original,
        "fitness": meta_fitness,
        "X_train": X_train_meta,
        "X_test": X_test_meta,
        "model": model,
        "y_pred": y_pred,
        "metrics": metrics
    }

    print(row)

metaheuristic_results_df = format_results_table(metaheuristic_results)
metaheuristic_results_df["Fitness"] = metaheuristic_results_df["Fitness"].astype(float).round(4)
metaheuristic_results_df = metaheuristic_results_df.sort_values(
    by=["F1 score", "Accuracy"],
    ascending=False
).reset_index(drop=True)

print("\nTabel hasil metaheuristik")
display(metaheuristic_results_df[["Base Statistical Method", "Metaheuristic", "Selected Features", "Precision", "Recall", "F1 score", "Accuracy"]])

BEST_METAHEURISTIC_METHOD = metaheuristic_results_df.loc[0, "Metaheuristic"]
print("Metaheuristik terbaik berdasarkan F1 score:", BEST_METAHEURISTIC_METHOD)


In [ ]:
# ============================================================
# 20. Evaluasi per label untuk model metaheuristik terbaik
# ============================================================
best_meta_artifact = metaheuristic_artifacts[BEST_METAHEURISTIC_METHOD]
Y_pred_best = best_meta_artifact["y_pred"]

precision_per_label = precision_score(Y_test, Y_pred_best, average=None, zero_division=0)
recall_per_label = recall_score(Y_test, Y_pred_best, average=None, zero_division=0)
f1_per_label = f1_score(Y_test, Y_pred_best, average=None, zero_division=0)
accuracy_per_label = [accuracy_score(Y_test[:, i], Y_pred_best[:, i]) for i in range(len(LABELS))]

per_label_df = pd.DataFrame({
    "Label": LABELS,
    "Precision": precision_per_label,
    "Recall": recall_per_label,
    "F1 score": f1_per_label,
    "Accuracy": accuracy_per_label
})

for col in ["Precision", "Recall", "F1 score", "Accuracy"]:
    per_label_df[col] = per_label_df[col].round(4)

print("Evaluasi per label untuk model terbaik:")
print("Metode statistik:", BEST_STATISTICAL_METHOD)
print("Metaheuristik   :", BEST_METAHEURISTIC_METHOD)
display(per_label_df)


In [ ]:
# ============================================================
# 21. Simpan hasil evaluasi, fitur terpilih, dan model terbaik
# ============================================================
best_meta_model = best_meta_artifact["model"]
best_selected_original_idx = best_meta_artifact["selected_idx_original"]

model_package = {
    "embedding_backend": EMBEDDING_BACKEND,
    "model_name": MODEL_NAME,
    "tfidf_vectorizer": TFIDF_VECTORIZER,
    "labels": LABELS,
    "text_column": TEXT_COL,
    "clean_text_column": CLEAN_TEXT_COL,
    "split_random_state": SPLIT_RANDOM_STATE,
    "best_statistical_method": BEST_STATISTICAL_METHOD,
    "best_metaheuristic_method": BEST_METAHEURISTIC_METHOD,
    "best_selected_original_idx": best_selected_original_idx,
    "classifier": best_meta_model,
    "statistical_results": statistical_results_df,
    "metaheuristic_results": metaheuristic_results_df,
    "per_label_results": per_label_df,
    "config": {
        "test_size": TEST_SIZE,
        "feature_top_k": FEATURE_TOP_K,
        "theta_vt": THETA_VT,
        "relief_sample_size": RELIEF_SAMPLE_SIZE,
        "rf_n_estimators": RF_N_ESTIMATORS,
        "rf_max_depth": RF_MAX_DEPTH,
        "cv_splits": CV_SPLITS,
        "meta_pop_size": META_POP_SIZE,
        "meta_max_iter": META_MAX_ITER,
        "meta_patience": META_PATIENCE,
        "random_state": RANDOM_STATE
    }
}

joblib.dump(model_package, "best_stat_meta_rf_multilabel_model.joblib")
statistical_results_df.to_csv("hasil_seleksi_fitur_statistik.csv", index=False)
metaheuristic_results_df.to_csv("hasil_metaheuristik.csv", index=False)
per_label_df.to_csv("hasil_per_label_model_terbaik.csv", index=False)
pd.DataFrame({"selected_feature_original_index": best_selected_original_idx}).to_csv(
    "fitur_terpilih_model_terbaik.csv",
    index=False
)

print("File berhasil disimpan:")
print("1. best_stat_meta_rf_multilabel_model.joblib")
print("2. hasil_seleksi_fitur_statistik.csv")
print("3. hasil_metaheuristik.csv")
print("4. hasil_per_label_model_terbaik.csv")
print("5. fitur_terpilih_model_terbaik.csv")


## Catatan Penggunaan

Jalankan notebook ini dari atas ke bawah. Jangan menjalankan cell bagian tengah terlebih dahulu, karena variabel seperti `X_train_emb`, `X_train_bal`, `Y_train_bal`, `X_train_best_stat`, dan `BEST_STATISTICAL_METHOD` harus terbentuk secara berurutan.

Untuk eksperimen final, nilai `FEATURE_TOP_K`, `META_POP_SIZE`, dan `META_MAX_ITER` dapat dinaikkan. Nilai awal dibuat sedang agar proses uji coba tidak terlalu berat.
